### Import modules and data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.cleaning import clean_string_series

In [ ]:
reservoir_path = PATHS['pre_EDA'] / 'detailed_reservoir_for_EDA.csv'
detailed_reservoirs_pd = pd.read_csv(reservoir_path)
detailed_reservoirs_pd.head()

### Correlations Matrix between Features

In [ ]:
correlation_matrix = detailed_reservoirs_pd.select_dtypes(include=[np.number]).corr()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    cmap='coolwarm',        
    annot=True,
    fmt='.2f',
    vmin=-1, vmax=1,
    center=0,
    linewidths=.5,
    cbar_kws={'label': 'Pearson r'}
)
plt.title('Correlation Matrix (Numeric Features)')
plt.tight_layout()
plt.show()

As we can see, there isn't a strong (linear) correlation between features


### Analyzing code column

In [ ]:
print(f"Out of {len(detailed_reservoirs_pd)} rows, there are {detailed_reservoirs_pd['code'].nunique()} code unique ones.")

We already have the id, so this column can be deleted

### Selection between name and reservoir column

We start getting those values that are different

In [ ]:
different_values = detailed_reservoirs_pd[(detailed_reservoirs_pd['name'] == detailed_reservoirs_pd['reservoir']) == False]
different_values.head()

Seems like reservoir has more useless stopwords, let's investigate further.

In [ ]:
print(f"Number of rows in 'name' column containing specific stopwords: {detailed_reservoirs_pd['name'].str.contains('reservoir|dam|embalse').sum()}\n")
print(f"Number of rows in 'reservoir' column containing specific stopwords: {detailed_reservoirs_pd['reservoir'].str.contains('reservoir|dam|embalse').sum()}\n")

Let's study how many rows of each column match the names from the reservoir dataframe

In [ ]:
reservoirs_path = PATHS['cleaned_data_notebooks'] / 'reservoirs_cleaned.csv'
reservoirs = pd.read_csv(reservoirs_path)
reservoirs.head()

In [ ]:
column_name = detailed_reservoirs_pd['name']
name_merged = pd.merge(column_name, reservoirs, left_on='name', right_on='name', how='inner')
print(f"Number of rows in 'name' column matching names from reservoir dataframe: {len(name_merged)}")

In [ ]:
column_reservoir = clean_string_series(detailed_reservoirs_pd['reservoir'].str.replace('reservoir|dam|embalse', '', regex=True))
reservoir_merged = pd.merge(column_reservoir, reservoirs, left_on='reservoir', right_on='name', how='inner')
print(f"Number of rows in 'reservoir' column matching names from reservoir dataframe: {len(reservoir_merged)}")

As there are more reservoirs matching with name column, reservoir will be dropped

### Study of correctness of string data

For the column basin:

In [ ]:
detailed_reservoirs_pd['basin'].unique()

They are all valid values.

For the column riverbed:

In [ ]:
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['riverbed'].str.contains('rio|arroyo|rivera|riu|riera|rambla|barranc', regex=True) == False,'riverbed'].unique()

As we can see, the first one is 'sin nombre', that means 'without name' in Spanish, so it will have to be converted to nan when cleaning the data.

The columns province and autonomous_community will be checked more deeply at merges.ipynb

The column type:

In [ ]:
detailed_reservoirs_pd['type'].unique()

Every value is correct in this case

### Study of Missing Values

In [ ]:
detailed_reservoirs_pd.isna().sum()

#### Riverbed Column

In [ ]:
non_nan_riverbed = detailed_reservoirs_pd[detailed_reservoirs_pd['riverbed'].notna()]
non_nan_riverbed_column = non_nan_riverbed['riverbed']
print(f"There are {len(non_nan_riverbed_column)} riverbeds, and {non_nan_riverbed_column.nunique()} are different")

Let's get those that are at least repeated 3 times

In [ ]:
non_nan_riverbed[non_nan_riverbed_column.map(non_nan_riverbed_column.value_counts() > 2)].sort_values('riverbed')

As we can observe, repeated riverbeds tend to have similar longitude and/or latitude values, so we will impute missing values based on the closest non-null riverbed.

### Crest Elevation Column

We carry out the same study as with riverbeds, in case there are reservoirs with the same exact crest elevation:

In [ ]:
non_nan_crest = detailed_reservoirs_pd[detailed_reservoirs_pd['crest_elevation'].notna()]
non_nan_crest_column = non_nan_crest['crest_elevation']
non_nan_crest[non_nan_crest_column.map(non_nan_crest_column.value_counts() > 2)].sort_values(by='crest_elevation')

What we can see here is that there are rows that are very similar, just differs a bit on the coordinates, but they represent the same reservoir, so this will also have to be tackled in the cleaning process.

In [ ]:
non_nan_crest[(non_nan_crest['longitude'] > 36) & (non_nan_crest['longitude'] < 37) & (non_nan_crest['latitude'] > -6) & (non_nan_crest['latitude'] < -5)][['name','crest_elevation','latitude','longitude']]


Here we see with an ilustrative example, a logical fact: closer reservoirs have similar crest elevation, so as we stated, they will be imputed using the closest known value

### Dam Height Column

We start by plotting a boxplot to learn about the distribution of dam height:

In [ ]:
detailed_reservoirs_pd['dam_height'].plot.box()

There is only one outlier, and half the reservoirs have a dam height between 40 and 80 meters

And now an histogram to see another type of distribution:

In [ ]:
detailed_reservoirs_pd['dam_height'].plot(kind='hist', bins=30, alpha=0.6)

- At modeling/missing_values_imputation.ipynb it has been tried to develop several types of Random Forest Regressors to impute missing values in the dam height column. 
- As the predictions turned out to be random, it was decided to leave the missing values as they were.
- Nevertheless, the type of reservoir is the only column to seem to have a significant impact on the dam height, so it could be used if needed:

In [ ]:
detailed_reservoirs_pd.boxplot(column='dam_height', by='type', rot=90)

So it could be imputed the median of the dam height by the type of reservoir if needed compulsory